# function-gemma-230m Agent Test

## Outline

 * Direct API calls to OpenAI API (custom model that enabled by llama.cpp's llama-server).
 * To see whether the custom model has agent capability.

## Setup

In [ ]:
%pip install --no-cache-dir -qU python-dotenv openai


In [12]:
from dotenv import load_dotenv
load_dotenv()

BASE_URL = "http://192.168.1.111:8887/v1"
MODEL = "function-gemma-230m"
DEFAULT_SYSTEM_PROMPT = "You are a model that can do function calling with the following functions:"

In [3]:
from openai import OpenAI

client = OpenAI(base_url=BASE_URL, api_key="sk-1")

def get_completion(prompt, tools=[], model=MODEL):
    messages = [
        { "role": "system", "content": DEFAULT_SYSTEM_PROMPT },
        { "role": "user", "content": prompt }
    ]

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools = tools,
        temperature=0.3, 
        max_tokens=1024
    )

    msg = response.choices[0].message
    if msg.content:
        return msg.content
    elif msg.tool_calls:
        return msg.tool_calls
    else:
        return None

In [4]:
get_completion("Calculate 1+1?")

'I can certainly calculate 1 + 1. Could you please specify the value you would like me to use?'

In [5]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather for a specified city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Name of the city"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

get_completion("What's the weather like in Boston right now?", tools=tools)

[ChatCompletionMessageFunctionToolCall(id='K0KVctRFtas41e3HWC71icpiydghFEyV', function=Function(arguments='{"city":"Boston"}', name='get_current_weather'), type='function')]

## Simple tool calling

In [7]:
import json
import requests
import sys

def test_simple_tool_call():
    """Simple direct test of tool calling with clear instructions"""
    print("=== Testing Simple Tool Call ===")
    
    # Define a calculator tool
    tools = [
        {
            "type": "function",
            "function": {
                "name": "calculate",
                "description": "Calculate a mathematical expression",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "expression": {
                            "type": "string",
                            "description": "The mathematical expression to calculate"
                        }
                    },
                    "required": ["expression"]
                }
            }
        }
    ]
    
    # Create the chat request with explicit instruction to use the tool
    request_data = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": "You are a helpful assistant. When asked to calculate something, ALWAYS use the calculate function."},
            {"role": "user", "content": "Calculate 123 + 456"}
        ],
        "tools": tools,
        "temperature": 0.1,  # Lower temperature for more deterministic outputs
        "stream": False
    }
    
    print("Sending request to calculate 123 + 456...")
    response = requests.post(f"{BASE_URL}/chat/completions", json=request_data)
    
    if response.status_code != 200:
        print(f"Error: {response.status_code}")
        print(response.text)
        return False
    
    response_data = response.json()
    # Check if tool_calls is in the response
    tool_calls = response_data["choices"][0].get("message", {}).get("tool_calls", [])
    has_tool_calls = len(tool_calls) > 0
    
    print(f"Has tool calls: {has_tool_calls}")
    
    if has_tool_calls:
        # Check if the function name is correct
        function_name = tool_calls[0].get("function", {}).get("name", "")
        print(f"Function called: {function_name}")
        
        # Check the arguments
        arguments = tool_calls[0].get("function", {}).get("arguments", "{}")
        if isinstance(arguments, str):
            try:
                arguments = json.loads(arguments)
            except:
                pass
        
        print(f"Arguments: {json.dumps(arguments, indent=2)}")
        
        # Check finish reason
        finish_reason = response_data["choices"][0].get("finish_reason")
        print(f"Finish reason: {finish_reason}")
    
    # Print the full response
    print("\nFull Response:")
    print(json.dumps(response_data, indent=2))
    
    return has_tool_calls

test_simple_tool_call()

=== Testing Simple Tool Call ===
Sending request to calculate 123 + 456...
Has tool calls: True
Function called: calculate
Arguments: {
  "expression": "123 + 456"
}
Finish reason: tool_calls

Full Response:
{
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "message": {
        "role": "assistant",
        "content": null,
        "tool_calls": [
          {
            "type": "function",
            "function": {
              "name": "calculate",
              "arguments": "{\"expression\":\"123 + 456\"}"
            },
            "id": "6O4yqeqymIKei827A03HvGRvZ8rzW9YY"
          }
        ]
      }
    }
  ],
  "created": 1766727560,
  "model": "function-gemma-230m",
  "system_fingerprint": "b2150-bc4064cf",
  "object": "chat.completion",
  "usage": {
    "completion_tokens": 29,
    "prompt_tokens": 123,
    "total_tokens": 152
  },
  "id": "chatcmpl-Z8iLIfjjAErc3bKvd1FmWvyuAcIAj9wK",
  "__verbose": {
    "index": 0,
    "content": "{\"tool_call

True

## More Tool Calling Tests

In [8]:
import time

def test_non_streaming_tool_call():
    """Test non-streaming tool calling"""
    print("\n=== Testing Non-Streaming Tool Calling ===")
    
    # Define a weather tool
    tools = [
        {
            "type": "function",
            "function": {
                "name": "get_current_weather",
                "description": "Get the current weather in a given location",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "location": {
                            "type": "string",
                            "description": "The city and state, e.g. San Francisco, CA"
                        },
                        "unit": {
                            "type": "string",
                            "enum": ["celsius", "fahrenheit"],
                            "description": "The temperature unit to use"
                        }
                    },
                    "required": ["location"]
                }
            }
        }
    ]
    
    # Create the chat request
    request_data = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "What's the weather like in New York City?"}
        ],
        "tools": tools,
        "stream": False
    }
    
    print("Sending request...")
    start_time = time.time()
    response = requests.post(f"{BASE_URL}/chat/completions", json=request_data)
    elapsed = time.time() - start_time
    
    if response.status_code != 200:
        print(f"Error: {response.status_code}")
        print(response.text)
        return False
    
    response_data = response.json()
    print(f"Response received in {elapsed:.2f} seconds")
    
    # Check if tool_calls is in the response
    has_tool_calls = "tool_calls" in response_data["choices"][0].get("message", {})
    print(f"Has tool calls: {has_tool_calls}")
    
    # Print the full response in a readable format
    print("\nResponse:")
    print(json.dumps(response_data, indent=2))
    
    return has_tool_calls

def test_streaming_tool_call():
    """Test streaming tool calling"""
    print("\n=== Testing Streaming Tool Calling ===")
    
    # Define a search tool
    tools = [
        {
            "type": "function",
            "function": {
                "name": "search_documents",
                "description": "Search for documents based on a query",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "query": {
                            "type": "string",
                            "description": "The search query"
                        },
                        "limit": {
                            "type": "integer",
                            "description": "Maximum number of results to return"
                        }
                    },
                    "required": ["query"]
                }
            }
        }
    ]
    
    # Create the chat request
    request_data = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "Search for documents about climate change"}
        ],
        "tools": tools,
        "stream": True
    }
    
    print("Sending streaming request...")
    start_time = time.time()
    response = requests.post(f"{BASE_URL}/chat/completions", json=request_data, stream=True)
    
    if response.status_code != 200:
        print(f"Error: {response.status_code}")
        print(response.text)
        return False
    
    print("Receiving stream events:")
    content_chunks = []
    tool_calls = []
    finish_reason = None
    
    try:
        for chunk in response.iter_lines():
            if chunk:
                # Filter out keep-alive new lines
                if chunk.startswith(b"data:"):
                    data_str = chunk[5:].decode('utf-8').strip()
                    if data_str != "[DONE]":
                        data = json.loads(data_str)
                        
                        # Check for content
                        delta = data["choices"][0]["delta"]
                        if "content" in delta:
                            content_chunks.append(delta["content"])
                            print(f"Content: {delta['content']}", end="", flush=True)
                        
                        # Check for tool calls
                        if "tool_calls" in delta:
                            tool_call = delta["tool_calls"][0]
                            
                            # Print tool call information
                            if "function" in tool_call:
                                function_info = tool_call["function"]
                                if "name" in function_info:
                                    print(f"\nTool call - Function: {function_info['name']}")
                                if "arguments" in function_info:
                                    args = json.loads(function_info['arguments'])
                                    print(f"Arguments: {json.dumps(args, indent=2)}")
                                    
                            tool_calls.append(tool_call)
                        
                        # Check for finish reason
                        if data["choices"][0]["finish_reason"] is not None:
                            finish_reason = data["choices"][0]["finish_reason"]
                            print(f"\nFinish reason: {finish_reason}")
    except Exception as e:
        print(f"Error parsing streaming response: {e}")
    
    elapsed = time.time() - start_time
    print(f"\nStreaming completed in {elapsed:.2f} seconds")
    
    has_tool_calls = len(tool_calls) > 0
    print(f"Has tool calls: {has_tool_calls}")
    print(f"Finish reason was: {finish_reason}")
    
    return has_tool_calls and finish_reason == "tool_calls"


def test_tool_choice():
    """Test tool_choice parameter to require a specific tool"""
    print("\n=== Testing Tool Choice Parameter ===")
    
    # Define multiple tools
    tools = [
        {
            "type": "function",
            "function": {
                "name": "get_current_weather",
                "description": "Get the current weather in a given location",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "location": {
                            "type": "string",
                            "description": "The city and state, e.g. San Francisco, CA"
                        }
                    },
                    "required": ["location"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "search_web",
                "description": "Search the web for information",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "query": {
                            "type": "string",
                            "description": "The search query"
                        }
                    },
                    "required": ["query"]
                }
            }
        }
    ]
    
    # Create the chat request with tool_choice
    request_data = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "What's the weather like in Paris?"}
        ],
        "tools": tools,
        "tool_choice": {"type": "function", "function": {"name": "get_current_weather"}},
        "stream": False
    }
    
    print("Sending request with tool_choice...")
    response = requests.post(f"{BASE_URL}/chat/completions", json=request_data)
    
    if response.status_code != 200:
        print(f"Error: {response.status_code}")
        print(response.text)
        return False
    
    response_data = response.json()
    
    # Check if the right tool was called
    tool_calls = response_data["choices"][0].get("message", {}).get("tool_calls", [])
    correct_tool_used = False
    
    if tool_calls:
        for tool_call in tool_calls:
            if tool_call.get("function", {}).get("name") == "get_current_weather":
                correct_tool_used = True
                break
    
    print(f"Used correct tool (get_current_weather): {correct_tool_used}")
    
    # Print the response
    print("\nResponse:")
    print(json.dumps(response_data, indent=2))
    
    return correct_tool_used

In [9]:
test_non_streaming_tool_call()


=== Testing Non-Streaming Tool Calling ===
Sending request...
Response received in 2.20 seconds
Has tool calls: True

Response:
{
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "message": {
        "role": "assistant",
        "content": null,
        "tool_calls": [
          {
            "type": "function",
            "function": {
              "name": "get_current_weather",
              "arguments": "{\"location\":\"New York City\"}"
            },
            "id": "hSQyN5BKzQkFPUPAQ6JyS8K86txTvDNQ"
          }
        ]
      }
    }
  ],
  "created": 1766727590,
  "model": "function-gemma-230m",
  "system_fingerprint": "b2150-bc4064cf",
  "object": "chat.completion",
  "usage": {
    "completion_tokens": 29,
    "prompt_tokens": 158,
    "total_tokens": 187
  },
  "id": "chatcmpl-2OGUQnN8mkyZF5xzfGQhQdtGNvTSuGz2",
  "__verbose": {
    "index": 0,
    "content": "{\"tool_call\": {\"name\": \"get_current_weather\", \"arguments\": {\"location\

True

In [10]:
test_streaming_tool_call()


=== Testing Streaming Tool Calling ===
Sending streaming request...
Receiving stream events:
Content: None
Tool call - Function: search_documents
Error parsing streaming response: Expecting value: line 1 column 1 (char 0)

Streaming completed in 1.87 seconds
Has tool calls: False
Finish reason was: None


False

In [11]:
test_tool_choice()


=== Testing Tool Choice Parameter ===
Sending request with tool_choice...
Used correct tool (get_current_weather): True

Response:
{
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "message": {
        "role": "assistant",
        "content": null,
        "tool_calls": [
          {
            "type": "function",
            "function": {
              "name": "get_current_weather",
              "arguments": "{\"location\":\"Paris\"}"
            },
            "id": "VnUXnpWSHG3ndKPRPZ3LiKL5sAzYX8Iw"
          }
        ]
      }
    }
  ],
  "created": 1766727611,
  "model": "function-gemma-230m",
  "system_fingerprint": "b2150-bc4064cf",
  "object": "chat.completion",
  "usage": {
    "completion_tokens": 26,
    "prompt_tokens": 176,
    "total_tokens": 202
  },
  "id": "chatcmpl-QCYvwT7rssG2iEth0s5FKDTGiWBuFRgP",
  "__verbose": {
    "index": 0,
    "content": "{\"tool_call\": {\"name\": \"get_current_weather\", \"arguments\": {\"location\": \"

True